# STRAT-412: Sprouts Farmers Market Store Directory Scraper

This notebook scrapes the entire Sprouts Farmers Market store directory from https://www.sprouts.com/stores/ and exports the data to a CSV file.

The scraper navigates a 3-level hierarchy:
1. Main directory → State pages
2. State pages → City pages
3. City pages → Individual store pages

**Output CSV columns:** Store Name, Store Number, Store Complex, Address, City, State, Zip, Phone Number

**Note:** The Sprouts website blocks standard HTTP requests (403 Forbidden), so we use Selenium with headless Chrome to render pages like a real browser.

## Imports & Setup
Install Selenium and ChromeDriver, then configure a headless Chrome browser. We also import BeautifulSoup to parse the rendered HTML that Selenium returns.

In [ ]:
# Install required packages for Google Colab
!pip install selenium beautifulsoup4 lxml
!apt-get update -qq
!apt-get install -y -qq chromium-browser chromium-chromedriver

import csv
import json
import time
import re
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Base URL for the Sprouts website
BASE_URL = "https://www.sprouts.com"

# Configure headless Chrome options for Colab
chrome_options = Options()
chrome_options.add_argument("--headless")              # Run without a GUI
chrome_options.add_argument("--no-sandbox")            # Required for Colab
chrome_options.add_argument("--disable-dev-shm-usage") # Avoid shared memory issues
chrome_options.add_argument("--disable-gpu")           # Disable GPU acceleration
chrome_options.add_argument("--window-size=1920,1080") # Set a standard window size
chrome_options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
)

# Initialize the Chrome driver
service = Service("/usr/bin/chromedriver")
driver = webdriver.Chrome(service=service, options=chrome_options)


def get_soup(url, wait_seconds=3):
    """
    Uses Selenium to load a page (bypassing 403 blocks), waits for content
    to render, then returns a BeautifulSoup object of the rendered HTML.
    """
    driver.get(url)
    time.sleep(wait_seconds)  # wait for JavaScript to render the page
    html = driver.page_source
    return BeautifulSoup(html, "lxml")


# Quick test to verify the driver works
driver.get("https://www.sprouts.com")
print(f"Page title: {driver.title}")
print("Setup complete! Selenium is working.")

## Discovery: Inspect Raw HTML
Run this cell to see the raw HTML structure of a store page as rendered by the browser. This helps verify where store name, address, phone, store number, etc. appear in the HTML so we can write correct selectors.

In [ ]:
# Inspect the rendered HTML of a store page to identify correct selectors
test_url = "https://www.sprouts.com/store/co/westminster/westminster/"
soup = get_soup(test_url, wait_seconds=5)

# Print the first 5000 characters to see the page structure
print("=== FIRST 5000 CHARS ===")
print(soup.prettify()[:5000])

# Also check for JSON-LD structured data (very useful for clean extraction)
print("\n=== JSON-LD STRUCTURED DATA ===")
json_ld_scripts = soup.find_all("script", type="application/ld+json")
for i, script in enumerate(json_ld_scripts):
    try:
        data = json.loads(script.string)
        print(f"\nJSON-LD block {i+1}:")
        print(json.dumps(data, indent=2)[:2000])
    except:
        print(f"Could not parse JSON-LD block {i+1}")

# Check for the h1 tag (store name)
print("\n=== H1 TAG ===")
h1 = soup.find("h1")
print(h1.get_text(strip=True) if h1 else "No h1 found")

# Check for address elements
print("\n=== ADDRESS ELEMENTS ===")
addr = soup.find("address")
print(addr.get_text(strip=True) if addr else "No <address> tag found")

## Code Block #1: Scrape Location Info for One Store
This function extracts all 8 required fields from a single Sprouts store page.

**Extraction strategy (in priority order):**
1. JSON-LD structured data (`<script type="application/ld+json">`) — most reliable
2. Schema.org microdata (`itemprop` attributes) — fallback
3. HTML elements (`<h1>`, `<address>` tags) — secondary fallback
4. Regex patterns — last resort for phone numbers and store numbers

In [ ]:
def scrape_one_store(url):
    """
    Scrapes a single Sprouts store page and returns a dictionary with:
    Store Name, Store Number, Store Complex, Address, City, State, Zip, Phone Number
    """
    soup = get_soup(url)

    # Initialize all fields with empty defaults
    store_name = ""
    store_number = ""
    store_complex = ""
    address = ""
    city = ""
    state = ""
    zipcode = ""
    phone = ""

    # --- Store Name ---
    # The store name is typically in the page's <h1> tag
    h1 = soup.find("h1")
    if h1:
        store_name = h1.get_text(strip=True)

    # --- Store Number ---
    # Look for \"Store #\" or \"Store No.\" patterns in the page text
    page_text = soup.get_text()
    store_num_match = re.search(r'Store\s*#?\s*(\d+)', page_text)
    if store_num_match:
        store_number = store_num_match.group(1)

    # --- Address, City, State, Zip, Phone ---
    # Try JSON-LD structured data first (most reliable if present)
    json_ld_scripts = soup.find_all("script", type="application/ld+json")
    for script in json_ld_scripts:
        try:
            data = json.loads(script.string)
            # Handle both single object and array formats
            if isinstance(data, list):
                data = data[0]
            # Extract address fields from JSON-LD
            if "address" in data:
                addr = data["address"]
                address = addr.get("streetAddress", "")
                city = addr.get("addressLocality", "")
                state = addr.get("addressRegion", "")
                zipcode = addr.get("postalCode", "")
            # Extract phone from JSON-LD
            if "telephone" in data:
                phone = data["telephone"]
            # Extract name from JSON-LD if not found in <h1>
            if "name" in data and not store_name:
                store_name = data["name"]
            # Some sites store branch/store number in branchCode
            if "branchCode" in data and not store_number:
                store_number = data["branchCode"]
        except (json.JSONDecodeError, TypeError, KeyError):
            continue

    # Fallback: look for address in <address> tag
    if not address:
        addr_tag = soup.find("address")
        if addr_tag:
            addr_text = addr_tag.get_text(separator=", ", strip=True)
            address = addr_text

    # Fallback: look for itemprop attributes (Schema.org microdata)
    if not address:
        street_el = soup.find(attrs={"itemprop": "streetAddress"})
        if street_el:
            address = street_el.get_text(strip=True)
    if not city:
        city_el = soup.find(attrs={"itemprop": "addressLocality"})
        if city_el:
            city = city_el.get_text(strip=True)
    if not state:
        state_el = soup.find(attrs={"itemprop": "addressRegion"})
        if state_el:
            state = state_el.get_text(strip=True)
    if not zipcode:
        zip_el = soup.find(attrs={"itemprop": "postalCode"})
        if zip_el:
            zipcode = zip_el.get_text(strip=True)

    # --- Phone Number ---
    if not phone:
        phone_el = soup.find(attrs={"itemprop": "telephone"})
        if phone_el:
            phone = phone_el.get_text(strip=True)
    if not phone:
        # Also try finding a tel: link
        tel_link = soup.find("a", href=re.compile(r'^tel:'))
        if tel_link:
            phone = tel_link.get_text(strip=True)
    if not phone:
        # Regex fallback: search for phone number pattern in page text
        phone_match = re.search(r'\(?\d{3}\)?[\s.-]?\d{3}[\s.-]?\d{4}', page_text)
        if phone_match:
            phone = phone_match.group(0)

    # --- Store Complex ---
    # The shopping center/plaza name may appear as a subtitle or specific class
    complex_el = (
        soup.find(class_=re.compile(r'complex|plaza|center|shopping', re.I))
        or soup.find(class_=re.compile(r'subtitle|subheading|sub-title', re.I))
    )
    if complex_el:
        store_complex = complex_el.get_text(strip=True)

    # Also check h2 elements for shopping center names
    if not store_complex:
        h2_tags = soup.find_all("h2")
        for h2 in h2_tags:
            text = h2.get_text(strip=True)
            # Shopping centers often contain these keywords
            if re.search(r'Plaza|Center|Village|Square|Mall|Shopping|Market|Commons', text, re.I):
                store_complex = text
                break

    # --- Data Cleaning ---
    # Ensure zip code is exactly 5 digits (trim any +4 extension)
    zip_match = re.search(r'\d{5}', zipcode)
    if zip_match:
        zipcode = zip_match.group(0)

    # Ensure state is a 2-character uppercase abbreviation
    state = state.strip().upper()[:2]

    # Clean phone number
    phone = phone.strip()

    return {
        "Store Name": store_name,
        "Store Number": store_number,
        "Store Complex": store_complex,
        "Address": address,
        "City": city,
        "State": state,
        "Zip": zipcode,
        "Phone Number": phone,
    }


# Test with one store to verify the function works
print("=" * 60)
print("SCRAPING ONE STORE")
print("=" * 60)
test_url = "https://www.sprouts.com/store/co/westminster/westminster/"
result = scrape_one_store(test_url)
for key, value in result.items():
    print(f"  {key}: {value}")

## Code Block #2: Test with a Small Loop of 3-4 URLs
Testing the scrape function on a few known store URLs from different states to verify it works correctly across different page layouts before scaling up to all ~487 stores.

In [ ]:
# Define 4 test store URLs from different states
test_urls = [
    "https://www.sprouts.com/store/co/westminster/westminster/",
    "https://www.sprouts.com/store/ga/atlanta/529-buford-i85-hwy20/",
    "https://www.sprouts.com/store/wa/seattle/seattle/",
    "https://www.sprouts.com/store/nj/cliffwood/hwy-35/",
]

print("=" * 60)
print("TESTING WITH SMALL LOOP (4 URLs)")
print("=" * 60)

test_results = []
for url in test_urls:
    print(f"\nScraping: {url}")
    try:
        store_data = scrape_one_store(url)
        test_results.append(store_data)
        for key, value in store_data.items():
            print(f"  {key}: {value}")
    except Exception as e:
        print(f"  ERROR: {e}")

print(f"\nSuccessfully scraped {len(test_results)} out of {len(test_urls)} test stores.")

## Code Block #3: Print the List of State URLs
Scrape the main store directory page to find all state-level URLs. State links follow the pattern `/stores/{state-abbreviation}/` (e.g., `/stores/ca/` for California).

Includes a hardcoded fallback list of 24 known Sprouts states in case the page structure is unexpected.

In [ ]:
print("=" * 60)
print("GETTING STATE URLs")
print("=" * 60)

# Fetch the main store directory page using Selenium
soup = get_soup(f"{BASE_URL}/stores/", wait_seconds=5)

# Find all links that match the state URL pattern: /stores/XX/
# State abbreviations are 2 lowercase letters
state_urls = []
for link in soup.find_all("a", href=True):
    href = link["href"]
    # Match pattern like /stores/az/ or /stores/ca/ (2-letter state codes)
    if re.match(r'^/stores/[a-z]{2}/?$', href):
        full_url = BASE_URL + href.rstrip("/") + "/"
        if full_url not in state_urls:
            state_urls.append(full_url)
    # Also handle full URLs (https://www.sprouts.com/stores/az/)
    elif re.match(r'^https?://www\.sprouts\.com/stores/[a-z]{2}/?$', href):
        full_url = href.rstrip("/") + "/"
        if full_url not in state_urls:
            state_urls.append(full_url)

# ---------- FALLBACK ----------
# If the directory page returns no state links, use this hardcoded list
# of all known Sprouts states as a backup.
if not state_urls:
    print("No state links found dynamically. Using hardcoded state list as fallback.")
    KNOWN_STATES = [
        "al", "az", "ca", "co", "de", "fl", "ga", "ks", "la", "md",
        "mo", "nv", "nj", "nm", "ny", "nc", "ok", "pa", "sc", "tn",
        "tx", "ut", "va", "wa",
    ]
    state_urls = [f"{BASE_URL}/stores/{s}/" for s in KNOWN_STATES]

# Sort alphabetically for readability
state_urls.sort()

print(f"Found {len(state_urls)} state URLs:\n")
for url in state_urls:
    print(f"  {url}")

## Code Block #4: Print the List of City URLs
For each state page, scrape the city-level links. City links follow the pattern `/stores/{state}/{city}/` (e.g., `/stores/ca/los-angeles/`).

In [ ]:
print("=" * 60)
print("GETTING CITY URLs")
print("=" * 60)

city_urls = []

for state_url in state_urls:
    print(f"Scraping cities from: {state_url}")
    try:
        soup = get_soup(state_url)

        for link in soup.find_all("a", href=True):
            href = link["href"]
            # Match city-level URLs: /stores/{state}/{city}/
            # State is 2 letters, city is one or more lowercase letters/hyphens
            if re.match(r'^/stores/[a-z]{2}/[\w-]+/?$', href):
                full_url = BASE_URL + href.rstrip("/") + "/"
                if full_url not in city_urls:
                    city_urls.append(full_url)
            # Also handle full URLs
            elif re.match(r'^https?://www\.sprouts\.com/stores/[a-z]{2}/[\w-]+/?$', href):
                full_url = href.rstrip("/") + "/"
                if full_url not in city_urls:
                    city_urls.append(full_url)
    except Exception as e:
        print(f"  ERROR scraping {state_url}: {e}")

city_urls.sort()

print(f"\nFound {len(city_urls)} city URLs:\n")
for url in city_urls:
    print(f"  {url}")

## Code Block #5: Print the List of Store URLs
For each city page (or state page if no cities exist), scrape the individual store page links.

**Important:** Individual store URLs use singular `/store/` (not `/stores/`) with the pattern `/store/{state}/{city}/{store-slug}/`.

In [ ]:
print("=" * 60)
print("GETTING STORE URLs")
print("=" * 60)

store_urls = []

def extract_store_links(soup):
    """Helper to find all individual store links in a page."""
    found = []
    for link in soup.find_all("a", href=True):
        href = link["href"]
        # Match individual store URLs: /store/{state}/{city}/{slug}/
        # Note the singular \"store\" (not \"stores\")
        if re.match(r'^/store/[a-z]{2}/[\w-]+/[\w-]+/?$', href):
            full_url = BASE_URL + href.rstrip("/") + "/"
            found.append(full_url)
        # Also handle full URLs
        elif re.match(r'^https?://www\.sprouts\.com/store/[a-z]{2}/[\w-]+/[\w-]+/?$', href):
            full_url = href.rstrip("/") + "/"
            found.append(full_url)
    return found

# If city URLs were found, scrape store links from each city page
# Otherwise, scrape store links directly from state pages
urls_to_scrape = city_urls if city_urls else state_urls

for page_url in urls_to_scrape:
    print(f"Scraping store links from: {page_url}")
    try:
        soup = get_soup(page_url)
        for url in extract_store_links(soup):
            if url not in store_urls:
                store_urls.append(url)
    except Exception as e:
        print(f"  ERROR scraping {page_url}: {e}")

# Also check state pages for direct store links (some states may list stores
# directly without an intermediate city page)
if city_urls:
    print("\nAlso checking state pages for direct store links...")
    for state_url in state_urls:
        try:
            soup = get_soup(state_url)
            for url in extract_store_links(soup):
                if url not in store_urls:
                    store_urls.append(url)
        except Exception as e:
            print(f"  ERROR scraping {state_url}: {e}")

store_urls.sort()

print(f"\nFound {len(store_urls)} store URLs:\n")
for url in store_urls:
    print(f"  {url}")

## Code Block #6: Loop Through All Store URLs
Scrape every individual store page and collect the data. This will take approximately 25-30 minutes (~487 stores × 3s per page). Progress is printed every 25 stores.

In [ ]:
print("=" * 60)
print("SCRAPING ALL STORES")
print("=" * 60)

all_stores = []
errors = []

total = len(store_urls)
for i, url in enumerate(store_urls, start=1):
    # Print progress every 25 stores
    if i % 25 == 0 or i == 1:
        print(f"  Progress: {i}/{total} stores scraped...")

    try:
        store_data = scrape_one_store(url)
        store_data["URL"] = url  # keep track of source URL for debugging
        all_stores.append(store_data)
    except Exception as e:
        print(f"  ERROR on {url}: {e}")
        errors.append({"url": url, "error": str(e)})

print(f"\nDone! Successfully scraped {len(all_stores)} stores.")
if errors:
    print(f"Encountered {len(errors)} errors:")
    for err in errors:
        print(f"  {err['url']}: {err['error']}")

## Code Block #7: Write to CSV and Export
Write the collected store data to a CSV file and download it to your local machine.

In [ ]:
csv_filename = "sprouts_stores.csv"
csv_columns = [
    "Store Name",
    "Store Number",
    "Store Complex",
    "Address",
    "City",
    "State",
    "Zip",
    "Phone Number",
]

# Write all store data to CSV
with open(csv_filename, "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=csv_columns, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(all_stores)

print(f"CSV file '{csv_filename}' written with {len(all_stores)} rows.")
print(f"Columns: {', '.join(csv_columns)}")

# Preview the first 5 rows
print("\nPreview (first 5 rows):")
for store in all_stores[:5]:
    print(f"  {store['Store Name']} | #{store['Store Number']} | "
          f"{store['Address']}, {store['City']}, {store['State']} {store['Zip']} | "
          f"{store['Phone Number']}")

# Clean up the Selenium driver
driver.quit()
print("\nSelenium driver closed.")

# Auto-download the CSV in Google Colab
from google.colab import files
files.download(csv_filename)